# CRM Enrichment Agent: Company Summaries & Suggested Next Actions

**Goal:** build an agent that takes raw CRM lead/contact records, enriches them with company intelligence (industry, size, recent news, tech stack), generates summaries, and proposes prioritized next actions for the sales team.

---

## Why CRM Enrichment?

Sales teams spend **30-40% of their time** on manual research before calls. A CRM enrichment agent automates this by:

1. **Looking up** company data from external sources (mocked here)
2. **Summarizing** key facts into a single-paragraph briefing
3. **Scoring** the lead based on fit signals
4. **Proposing** next actions ranked by priority and timing

## Data Flow & Tool Layers

The agent uses a **layered tool architecture** — each layer has a specific responsibility and a clear interface boundary.

```text
                    ┌─────────────────────────┐
  Layer 0: INPUT    │  Raw CRM Records (CSV)  │
                    └──────────┬──────────────┘
                               │
                    ┌──────────▼──────────────┐
  Layer 1: LOOKUP   │  Company Lookup Tool    │  domain → company profile
                    │  News Lookup Tool       │  domain → recent news
                    │  Tech Stack Tool        │  domain → technologies used
                    └──────────┬──────────────┘
                               │ enriched data
                    ┌──────────▼──────────────┐
  Layer 2: ANALYZE  │  Lead Scorer            │  profile → fit score (0-100)
                    │  Engagement Analyzer    │  CRM history → engagement level
                    └──────────┬──────────────┘
                               │ scores + signals
                    ┌──────────▼──────────────┐
  Layer 3: GENERATE │  Summary Generator      │  profile + scores → briefing
                    │  Action Recommender     │  scores + context → next steps
                    └──────────┬──────────────┘
                               │ enriched record
                    ┌──────────▼──────────────┐
  Layer 4: OUTPUT   │  Enriched CRM Record    │  JSON / DataFrame
                    └─────────────────────────┘
```

### Why Layers?

| Principle | Benefit |
|---|---|
| **Separation of concerns** | Each tool does one thing well |
| **Independent testing** | Test lookup without generation |
| **Swappable backends** | Replace mock with real API without changing agent |
| **Clear data flow** | Easy to debug which layer failed |
| **Composability** | Reuse lookup tools in other agents |


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

In [ ]:
import json
import re
import textwrap
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifact dir: {ARTIFACT_DIR}")

## 2. Data Model — CRM Records & Enrichment Outputs

Every structure the agent produces is a **typed dataclass**. This makes the data flow traceable and every intermediate result inspectable.


In [ ]:
class LeadStage(Enum):
    NEW = "new"
    CONTACTED = "contacted"
    QUALIFIED = "qualified"
    PROPOSAL = "proposal"
    NEGOTIATION = "negotiation"
    CLOSED_WON = "closed_won"
    CLOSED_LOST = "closed_lost"

    def __str__(self):
        return self.value


class ActionType(Enum):
    SEND_EMAIL = "send_email"
    SCHEDULE_CALL = "schedule_call"
    SEND_CASE_STUDY = "send_case_study"
    SCHEDULE_DEMO = "schedule_demo"
    FOLLOW_UP = "follow_up"
    ESCALATE = "escalate"
    NURTURE = "nurture"
    RE_ENGAGE = "re_engage"
    SEND_PROPOSAL = "send_proposal"
    INTRO_PARTNER = "introduce_partner"

    def __str__(self):
        return self.value


@dataclass
class CRMRecord:
    """Raw CRM lead/contact record."""
    lead_id: str
    contact_name: str
    contact_title: str
    company_name: str
    company_domain: str
    email: str
    stage: LeadStage
    deal_value: float           # Estimated deal size in USD
    last_contact_date: str      # ISO date
    notes: str                  # Free-text CRM notes
    source: str                 # "inbound", "outreach", "referral", "event"


@dataclass
class CompanyProfile:
    """Enriched company information from lookup tools."""
    domain: str
    name: str
    industry: str
    sub_industry: str
    employee_count: int
    revenue_estimate: str       # e.g. "$10M-$50M"
    founded_year: int
    headquarters: str
    description: str
    funding_stage: str          # "bootstrapped", "seed", "series_a", etc.
    recent_news: list[str]
    tech_stack: list[str]
    competitors: list[str]
    social_links: dict          # {"linkedin": url, "twitter": url}


@dataclass
class LeadScore:
    """Computed lead score with breakdown."""
    total_score: int            # 0-100
    fit_score: int              # Company fit (0-50)
    engagement_score: int       # Activity level (0-30)
    timing_score: int           # Urgency signals (0-20)
    signals: list[str]          # What drove the score
    tier: str                   # "hot", "warm", "cold"


@dataclass
class SuggestedAction:
    """A recommended next action."""
    action_type: ActionType
    priority: int               # 1 (highest) to 5 (lowest)
    description: str
    reasoning: str
    deadline_days: int          # Suggested days from now
    template: str               # Draft email/message if applicable


@dataclass
class CompanySummary:
    """Generated executive briefing for a lead."""
    headline: str               # One-line summary
    briefing: str               # 3-4 sentence paragraph
    key_facts: list[str]        # Bullet points
    talking_points: list[str]   # For the sales call
    risk_factors: list[str]     # Potential blockers


@dataclass
class EnrichedRecord:
    """Complete enriched CRM record — the agent's final output."""
    lead: CRMRecord
    profile: CompanyProfile
    score: LeadScore
    summary: CompanySummary
    actions: list[SuggestedAction]
    enriched_at: str = ""

    def __post_init__(self):
        if not self.enriched_at:
            self.enriched_at = datetime.now().isoformat()

    def to_dict(self) -> dict:
        d = asdict(self)
        d["lead"]["stage"] = str(self.lead.stage)
        for a in d["actions"]:
            a["action_type"] = str(a["action_type"])
        return d


print("Data model loaded.")
print(f"  LeadStage options:  {[s.value for s in LeadStage]}")
print(f"  ActionType options: {[a.value for a in ActionType]}")

## 3. Synthetic CRM Dataset

We create 10 realistic CRM records representing different stages, industries, and deal sizes.


In [ ]:
CRM_RECORDS = [
    CRMRecord(
        lead_id="LEAD-001", contact_name="Sarah Chen", contact_title="VP Engineering",
        company_name="TechFlow Inc", company_domain="techflow.io",
        email="sarah@techflow.io", stage=LeadStage.QUALIFIED, deal_value=85000,
        last_contact_date="2024-06-10",
        notes="Interested in our analytics platform. Currently using a competitor. Team of 50 engineers. Mentioned budget approval in Q3.",
        source="inbound",
    ),
    CRMRecord(
        lead_id="LEAD-002", contact_name="James Rodriguez", contact_title="CTO",
        company_name="DataPrime Solutions", company_domain="dataprime.com",
        email="james@dataprime.com", stage=LeadStage.NEW, deal_value=120000,
        last_contact_date="2024-06-14",
        notes="Downloaded whitepaper on ML ops. No direct contact yet. Company is growing fast.",
        source="inbound",
    ),
    CRMRecord(
        lead_id="LEAD-003", contact_name="Maria Gonzalez", contact_title="Director of Operations",
        company_name="RetailMax", company_domain="retailmax.com",
        email="maria@retailmax.com", stage=LeadStage.PROPOSAL, deal_value=200000,
        last_contact_date="2024-06-08",
        notes="Proposal sent Jun 5. Waiting for procurement review. Champion is strong. Legal review pending. They want to launch before Black Friday.",
        source="referral",
    ),
    CRMRecord(
        lead_id="LEAD-004", contact_name="Alex Kim", contact_title="Head of Data",
        company_name="FinServe Capital", company_domain="finserve.com",
        email="alex@finserve.com", stage=LeadStage.CONTACTED, deal_value=300000,
        last_contact_date="2024-05-20",
        notes="Had intro call May 15. Very interested but concerned about compliance. Need to address SOC2 and GDPR. No response to follow-up email.",
        source="outreach",
    ),
    CRMRecord(
        lead_id="LEAD-005", contact_name="Priya Patel", contact_title="CEO",
        company_name="GreenLogistics", company_domain="greenlogistics.co",
        email="priya@greenlogistics.co", stage=LeadStage.NEW, deal_value=50000,
        last_contact_date="2024-06-13",
        notes="Met at SaaS conference. Small company but fast-growing. Interested in route optimization module.",
        source="event",
    ),
    CRMRecord(
        lead_id="LEAD-006", contact_name="David Williams", contact_title="VP Sales",
        company_name="CloudNine Software", company_domain="cloudnine.dev",
        email="david@cloudnine.dev", stage=LeadStage.NEGOTIATION, deal_value=175000,
        last_contact_date="2024-06-12",
        notes="Negotiating pricing. They want 20% discount for annual commitment. Decision by end of June. Competing with Acme Corp offering.",
        source="outreach",
    ),
    CRMRecord(
        lead_id="LEAD-007", contact_name="Lena Mueller", contact_title="Product Manager",
        company_name="HealthBridge", company_domain="healthbridge.eu",
        email="lena@healthbridge.eu", stage=LeadStage.QUALIFIED, deal_value=95000,
        last_contact_date="2024-06-01",
        notes="Strong product fit. HIPAA compliance is a must. Currently evaluating 3 vendors. Demo scheduled for next week was postponed.",
        source="inbound",
    ),
    CRMRecord(
        lead_id="LEAD-008", contact_name="Tom Baker", contact_title="IT Director",
        company_name="MegaManufacturing", company_domain="megamfg.com",
        email="tom@megamfg.com", stage=LeadStage.CONTACTED, deal_value=500000,
        last_contact_date="2024-04-15",
        notes="Large enterprise. Slow procurement. Initial interest in IoT analytics. Haven't heard back in 2 months.",
        source="outreach",
    ),
    CRMRecord(
        lead_id="LEAD-009", contact_name="Aisha Johnson", contact_title="Marketing Director",
        company_name="BrandSpark Agency", company_domain="brandspark.co",
        email="aisha@brandspark.co", stage=LeadStage.CLOSED_LOST, deal_value=40000,
        last_contact_date="2024-03-20",
        notes="Lost to competitor in March. Mentioned they might reconsider in 6 months when contract is up for renewal.",
        source="referral",
    ),
    CRMRecord(
        lead_id="LEAD-010", contact_name="Ryan Park", contact_title="Chief Revenue Officer",
        company_name="ScaleUp Ventures", company_domain="scaleupvc.com",
        email="ryan@scaleupvc.com", stage=LeadStage.QUALIFIED, deal_value=150000,
        last_contact_date="2024-06-11",
        notes="Portfolio company referral. Interested in our reporting suite for their portfolio companies. Strategic partnership potential.",
        source="referral",
    ),
]

crm_df = pd.DataFrame([asdict(r) for r in CRM_RECORDS])
crm_df["stage"] = crm_df["stage"].apply(lambda s: s if isinstance(s, str) else str(s))
print(f"CRM dataset: {len(CRM_RECORDS)} records\n")
print(crm_df[["lead_id", "contact_name", "company_name", "stage", "deal_value", "source"]].to_string(index=False))

## 4. Layer 1 — Lookup Tools

Layer 1 tools fetch external data about a company. In production these would call APIs like Clearbit, Crunchbase, or LinkedIn. Here we use **mocked databases** with realistic data.

### Tool Design Principles

1. **Single responsibility** — one tool per data source
2. **Domain-keyed** — every lookup uses the company domain as the key
3. **Graceful degradation** — missing data returns defaults, not errors
4. **Typed returns** — every tool returns a typed object, not a raw dict


In [ ]:
# ── Mocked company database ──
COMPANY_DB: dict[str, dict] = {
    "techflow.io": {
        "name": "TechFlow Inc",
        "industry": "Technology",
        "sub_industry": "Developer Tools",
        "employee_count": 120,
        "revenue_estimate": "$10M-$50M",
        "founded_year": 2019,
        "headquarters": "San Francisco, CA",
        "description": "TechFlow builds developer productivity tools for engineering teams. Their platform integrates CI/CD, code review, and project management into a unified workflow.",
        "funding_stage": "series_b",
        "competitors": ["GitLab", "LinearB", "Jellyfish"],
        "social_links": {"linkedin": "linkedin.com/company/techflow", "twitter": "@techflow_io"},
    },
    "dataprime.com": {
        "name": "DataPrime Solutions",
        "industry": "Technology",
        "sub_industry": "Data Infrastructure",
        "employee_count": 250,
        "revenue_estimate": "$50M-$100M",
        "founded_year": 2017,
        "headquarters": "Austin, TX",
        "description": "DataPrime provides enterprise data management solutions including data catalogs, lineage tracking, and automated governance. Rapid growth in the financial services sector.",
        "funding_stage": "series_c",
        "competitors": ["Alation", "Collibra", "Atlan"],
        "social_links": {"linkedin": "linkedin.com/company/dataprime"},
    },
    "retailmax.com": {
        "name": "RetailMax",
        "industry": "Retail",
        "sub_industry": "E-commerce Technology",
        "employee_count": 800,
        "revenue_estimate": "$100M-$500M",
        "founded_year": 2012,
        "headquarters": "New York, NY",
        "description": "RetailMax is a leading e-commerce platform serving mid-market retailers. They handle inventory management, order fulfillment, and customer analytics for 2,000+ merchants.",
        "funding_stage": "series_d",
        "competitors": ["Shopify Plus", "BigCommerce", "Salesforce Commerce"],
        "social_links": {"linkedin": "linkedin.com/company/retailmax", "twitter": "@retailmax"},
    },
    "finserve.com": {
        "name": "FinServe Capital",
        "industry": "Financial Services",
        "sub_industry": "Wealth Management",
        "employee_count": 500,
        "revenue_estimate": "$50M-$100M",
        "founded_year": 2010,
        "headquarters": "Chicago, IL",
        "description": "FinServe Capital is a wealth management firm serving high-net-worth individuals and institutional investors. Manages $5B+ in assets. Heavy focus on regulatory compliance.",
        "funding_stage": "private_equity",
        "competitors": ["Betterment for Business", "Wealthfront", "Personal Capital"],
        "social_links": {"linkedin": "linkedin.com/company/finserve"},
    },
    "greenlogistics.co": {
        "name": "GreenLogistics",
        "industry": "Logistics",
        "sub_industry": "Sustainable Supply Chain",
        "employee_count": 45,
        "revenue_estimate": "$1M-$10M",
        "founded_year": 2021,
        "headquarters": "Portland, OR",
        "description": "GreenLogistics optimizes last-mile delivery with carbon-neutral routing algorithms. Focuses on urban areas with electric vehicle fleets. Series A just closed.",
        "funding_stage": "series_a",
        "competitors": ["Routific", "OptimoRoute", "Wise Systems"],
        "social_links": {"linkedin": "linkedin.com/company/greenlogistics"},
    },
    "cloudnine.dev": {
        "name": "CloudNine Software",
        "industry": "Technology",
        "sub_industry": "Cloud Infrastructure",
        "employee_count": 180,
        "revenue_estimate": "$10M-$50M",
        "founded_year": 2018,
        "headquarters": "Seattle, WA",
        "description": "CloudNine builds multi-cloud management tools for DevOps teams. Platform supports AWS, Azure, and GCP with unified cost optimization and security posture management.",
        "funding_stage": "series_b",
        "competitors": ["Terraform Cloud", "Pulumi", "Env0"],
        "social_links": {"linkedin": "linkedin.com/company/cloudnine"},
    },
    "healthbridge.eu": {
        "name": "HealthBridge",
        "industry": "Healthcare",
        "sub_industry": "Health Tech",
        "employee_count": 300,
        "revenue_estimate": "$10M-$50M",
        "founded_year": 2016,
        "headquarters": "Berlin, Germany",
        "description": "HealthBridge connects hospitals, clinics, and insurers through a unified patient data platform. HIPAA and GDPR compliant. Strong presence in DACH region.",
        "funding_stage": "series_b",
        "competitors": ["Epic Systems", "Cerner", "Veeva"],
        "social_links": {"linkedin": "linkedin.com/company/healthbridge"},
    },
    "megamfg.com": {
        "name": "MegaManufacturing",
        "industry": "Manufacturing",
        "sub_industry": "Industrial IoT",
        "employee_count": 5000,
        "revenue_estimate": "$500M-$1B",
        "founded_year": 1985,
        "headquarters": "Detroit, MI",
        "description": "MegaManufacturing is a legacy industrial manufacturer with 15 factories globally. Currently undergoing digital transformation. Investing heavily in IoT and predictive maintenance.",
        "funding_stage": "public",
        "competitors": ["Siemens", "Honeywell", "GE Digital"],
        "social_links": {"linkedin": "linkedin.com/company/megamfg"},
    },
    "brandspark.co": {
        "name": "BrandSpark Agency",
        "industry": "Marketing",
        "sub_industry": "Digital Marketing",
        "employee_count": 30,
        "revenue_estimate": "$1M-$10M",
        "founded_year": 2020,
        "headquarters": "Los Angeles, CA",
        "description": "BrandSpark is a digital marketing agency specializing in D2C brands. Services include paid social, influencer marketing, and creative production.",
        "funding_stage": "bootstrapped",
        "competitors": ["VaynerMedia", "Tinuiti", "Wpromote"],
        "social_links": {"linkedin": "linkedin.com/company/brandspark"},
    },
    "scaleupvc.com": {
        "name": "ScaleUp Ventures",
        "industry": "Financial Services",
        "sub_industry": "Venture Capital",
        "employee_count": 25,
        "revenue_estimate": "$10M-$50M",
        "founded_year": 2015,
        "headquarters": "Boston, MA",
        "description": "ScaleUp Ventures is a B2B SaaS-focused VC firm with $200M AUM across 3 funds. Portfolio of 40+ companies. Known for strong operational support of portfolio companies.",
        "funding_stage": "fund_iii",
        "competitors": ["a16z", "Bessemer", "Accel"],
        "social_links": {"linkedin": "linkedin.com/company/scaleupvc"},
    },
}


# ── Mocked news database ──
NEWS_DB: dict[str, list[str]] = {
    "techflow.io": [
        "TechFlow raises $30M Series B led by Sequoia (Jun 2024)",
        "TechFlow launches AI-powered code review feature (May 2024)",
        "Named in Gartner's Cool Vendors for DevOps (Apr 2024)",
    ],
    "dataprime.com": [
        "DataPrime expands to European market with Berlin office (Jun 2024)",
        "Partners with Snowflake for native data catalog integration (May 2024)",
        "DataPrime CEO featured in Forbes 30 Under 30 list (Jan 2024)",
    ],
    "retailmax.com": [
        "RetailMax acquires logistics startup ShipFast for $45M (Jun 2024)",
        "Launches AI-powered demand forecasting module (May 2024)",
        "Q1 2024 revenue up 35% year-over-year (Apr 2024)",
    ],
    "finserve.com": [
        "FinServe passes $5B AUM milestone (May 2024)",
        "Launches compliance automation suite for RIAs (Apr 2024)",
        "Hires former SEC examiner as Chief Compliance Officer (Mar 2024)",
    ],
    "greenlogistics.co": [
        "GreenLogistics closes $8M Series A (Jun 2024)",
        "Partners with municipal EV charging network in Portland (May 2024)",
        "Named 'Startup to Watch' by Logistics Quarterly (Apr 2024)",
    ],
    "cloudnine.dev": [
        "CloudNine adds GCP support, completing multi-cloud coverage (Jun 2024)",
        "Achieves SOC 2 Type II certification (May 2024)",
        "Wins 'Best DevOps Tool' at CloudExpo 2024 (Apr 2024)",
    ],
    "healthbridge.eu": [
        "HealthBridge achieves HIPAA compliance certification (Jun 2024)",
        "Expands to Swiss market with UBS health partnership (May 2024)",
        "Raises $25M Series B from health-focused fund (Mar 2024)",
    ],
    "megamfg.com": [
        "MegaManufacturing announces $100M digital transformation initiative (Jun 2024)",
        "Partners with AWS for industrial IoT platform (May 2024)",
        "Reports 15% efficiency gains from predictive maintenance pilot (Apr 2024)",
    ],
    "brandspark.co": [
        "BrandSpark wins Adweek's Small Agency of the Year (May 2024)",
        "Launches in-house AI content generation tool (Apr 2024)",
    ],
    "scaleupvc.com": [
        "ScaleUp Ventures closes Fund III at $80M (Jun 2024)",
        "Portfolio company acquired by Microsoft for $200M (May 2024)",
        "Launches CRO-as-a-Service for portfolio companies (Apr 2024)",
    ],
}


# ── Mocked tech stack database ──
TECH_DB: dict[str, list[str]] = {
    "techflow.io":      ["React", "Python", "Kubernetes", "PostgreSQL", "Redis", "GitHub Actions"],
    "dataprime.com":    ["Java", "Spark", "Snowflake", "React", "Terraform", "Datadog"],
    "retailmax.com":    ["Ruby on Rails", "React", "AWS", "MySQL", "Elasticsearch", "Stripe"],
    "finserve.com":     ["Java", ".NET", "Oracle DB", "Tableau", "Salesforce", "Azure"],
    "greenlogistics.co":["Python", "FastAPI", "PostgreSQL", "React Native", "GCP", "MapBox"],
    "cloudnine.dev":    ["Go", "TypeScript", "React", "Terraform", "AWS", "GCP", "Azure"],
    "healthbridge.eu":  ["Python", "Django", "PostgreSQL", "React", "AWS", "Docker", "FHIR"],
    "megamfg.com":      ["SAP", "Oracle", ".NET", "Azure IoT Hub", "Power BI", "SCADA"],
    "brandspark.co":    ["WordPress", "Shopify", "Google Ads API", "Meta Ads API", "Figma"],
    "scaleupvc.com":    ["React", "Node.js", "PostgreSQL", "Stripe", "HubSpot"],
}

print(f"Mocked databases loaded:")
print(f"  Company profiles: {len(COMPANY_DB)}")
print(f"  News entries:     {sum(len(v) for v in NEWS_DB.values())}")
print(f"  Tech stacks:      {len(TECH_DB)}")

In [ ]:
class ToolRegistry:
    """Layer 1 tool registry with call logging."""

    def __init__(self):
        self._call_log: list[dict] = []

    def _log(self, tool: str, domain: str, success: bool):
        self._call_log.append({
            "timestamp": datetime.now().isoformat(),
            "tool": tool,
            "domain": domain,
            "success": success,
        })

    def lookup_company(self, domain: str) -> CompanyProfile:
        """Layer 1 Tool: look up company profile by domain."""
        data = COMPANY_DB.get(domain)
        self._log("lookup_company", domain, data is not None)

        if not data:
            return CompanyProfile(
                domain=domain, name=domain.split(".")[0].title(),
                industry="Unknown", sub_industry="Unknown",
                employee_count=0, revenue_estimate="Unknown",
                founded_year=0, headquarters="Unknown",
                description="No company data available.",
                funding_stage="unknown", recent_news=[], tech_stack=[],
                competitors=[], social_links={},
            )

        return CompanyProfile(
            domain=domain,
            name=data["name"],
            industry=data["industry"],
            sub_industry=data["sub_industry"],
            employee_count=data["employee_count"],
            revenue_estimate=data["revenue_estimate"],
            founded_year=data["founded_year"],
            headquarters=data["headquarters"],
            description=data["description"],
            funding_stage=data["funding_stage"],
            recent_news=NEWS_DB.get(domain, []),
            tech_stack=TECH_DB.get(domain, []),
            competitors=data.get("competitors", []),
            social_links=data.get("social_links", {}),
        )

    def lookup_news(self, domain: str) -> list[str]:
        """Layer 1 Tool: fetch recent news for a company."""
        news = NEWS_DB.get(domain, [])
        self._log("lookup_news", domain, len(news) > 0)
        return news

    def lookup_tech_stack(self, domain: str) -> list[str]:
        """Layer 1 Tool: identify technologies used by a company."""
        stack = TECH_DB.get(domain, [])
        self._log("lookup_tech_stack", domain, len(stack) > 0)
        return stack

    @property
    def call_log(self):
        return self._call_log


tools = ToolRegistry()

# Test lookups
profile = tools.lookup_company("techflow.io")
print(f"Company: {profile.name}")
print(f"Industry: {profile.industry} / {profile.sub_industry}")
print(f"Employees: {profile.employee_count}")
print(f"Funding: {profile.funding_stage}")
print(f"Tech stack: {profile.tech_stack}")
print(f"Recent news: {profile.recent_news[0]}")

## 5. Layer 2 — Lead Scoring & Engagement Analysis

Layer 2 tools **analyze** the enriched data to produce scores. They receive typed inputs from Layer 1 and produce typed outputs for Layer 3.

### Scoring Methodology

| Factor | Weight | Signal examples |
|---|---|---|
| **Company fit** (0-50) | 50% | Industry match, employee count, funding stage, revenue |
| **Engagement** (0-30) | 30% | Recency of contact, CRM stage, source quality |
| **Timing** (0-20) | 20% | News triggers, urgency signals in notes, seasonal factors |


In [ ]:
# ── Industry fit mapping: how well does each industry match our ICP? ──
INDUSTRY_FIT = {
    "Technology": 10,
    "Financial Services": 8,
    "Healthcare": 7,
    "Retail": 6,
    "Logistics": 5,
    "Manufacturing": 4,
    "Marketing": 3,
}

# ── Employee count fit (sweet spot: 100-1000) ──
def employee_fit(count: int) -> int:
    if 100 <= count <= 1000:
        return 10
    elif 50 <= count < 100 or 1000 < count <= 2000:
        return 7
    elif 20 <= count < 50 or 2000 < count <= 5000:
        return 5
    elif count > 5000:
        return 3
    return 2


# ── Funding stage fit ──
FUNDING_FIT = {
    "series_b": 10, "series_c": 9, "series_a": 8,
    "series_d": 7, "private_equity": 6, "public": 5,
    "fund_iii": 5, "seed": 4, "bootstrapped": 3, "unknown": 1,
}


def score_lead(record: CRMRecord, profile: CompanyProfile) -> LeadScore:
    """Layer 2 Tool: compute a multi-factor lead score."""
    signals = []

    # ── Fit score (0-50) ──
    industry_pts = INDUSTRY_FIT.get(profile.industry, 2)
    emp_pts = employee_fit(profile.employee_count)
    funding_pts = FUNDING_FIT.get(profile.funding_stage, 2)

    # Revenue tier bonus
    rev_pts = 0
    if "$100M" in profile.revenue_estimate or "$500M" in profile.revenue_estimate:
        rev_pts = 10
        signals.append("High revenue tier")
    elif "$50M" in profile.revenue_estimate:
        rev_pts = 7
        signals.append("Mid-high revenue tier")
    elif "$10M" in profile.revenue_estimate:
        rev_pts = 5
        signals.append("Growth-stage revenue")
    else:
        rev_pts = 2

    fit_raw = industry_pts + emp_pts + funding_pts + rev_pts
    fit_score = min(int(fit_raw * 50 / 40), 50)
    signals.append(f"Industry fit: {profile.industry} ({industry_pts}/10)")
    signals.append(f"Company size: {profile.employee_count} employees ({emp_pts}/10)")

    # ── Engagement score (0-30) ──
    stage_pts = {
        LeadStage.NEGOTIATION: 30, LeadStage.PROPOSAL: 25,
        LeadStage.QUALIFIED: 20, LeadStage.CONTACTED: 12,
        LeadStage.NEW: 8, LeadStage.CLOSED_WON: 30,
        LeadStage.CLOSED_LOST: 3,
    }.get(record.stage, 5)

    # Recency bonus
    days_since = (date(2024, 6, 15) - date.fromisoformat(record.last_contact_date)).days
    recency_pts = max(0, 10 - days_since // 3)  # Lose points for every 3 days of silence
    if days_since > 30:
        signals.append(f"STALE: {days_since} days since last contact")
    elif days_since <= 7:
        signals.append("Recent engagement (past week)")

    # Source quality
    source_pts = {"referral": 5, "inbound": 4, "event": 3, "outreach": 2}.get(record.source, 1)
    signals.append(f"Lead source: {record.source}")

    engagement_raw = stage_pts + recency_pts + source_pts
    engagement_score = min(int(engagement_raw * 30 / 45), 30)

    # ── Timing score (0-20) ──
    timing = 5  # Baseline

    # Check notes for urgency signals
    notes_lower = record.notes.lower()
    urgency_keywords = {
        "budget approval": 4, "decision by": 4, "before black friday": 5,
        "launch before": 4, "contract is up": 3, "growing fast": 2,
        "digital transformation": 3, "partnership potential": 2,
        "currently evaluating": 3, "competing with": 3,
    }
    for keyword, pts in urgency_keywords.items():
        if keyword in notes_lower:
            timing += pts
            signals.append(f"Timing signal: '{keyword}'")

    # News-based timing
    for news in profile.recent_news:
        if any(kw in news.lower() for kw in ["raises", "closes", "acquires", "launches"]):
            timing += 2
            signals.append(f"Recent news trigger: {news[:50]}")
            break

    timing_score = min(timing, 20)

    # ── Total ──
    total = fit_score + engagement_score + timing_score
    tier = "hot" if total >= 65 else ("warm" if total >= 40 else "cold")

    return LeadScore(
        total_score=total,
        fit_score=fit_score,
        engagement_score=engagement_score,
        timing_score=timing_score,
        signals=signals,
        tier=tier,
    )


# Test scoring
test_profile = tools.lookup_company("techflow.io")
test_score = score_lead(CRM_RECORDS[0], test_profile)
print(f"Lead: {CRM_RECORDS[0].contact_name} @ {CRM_RECORDS[0].company_name}")
print(f"Score: {test_score.total_score}/100 ({test_score.tier})")
print(f"  Fit:        {test_score.fit_score}/50")
print(f"  Engagement: {test_score.engagement_score}/30")
print(f"  Timing:     {test_score.timing_score}/20")
print(f"  Signals:")
for s in test_score.signals:
    print(f"    - {s}")

## 6. Layer 3 — Summary Generation & Action Recommendation

Layer 3 consumes the scored, enriched data and produces **human-readable outputs**: an executive briefing and a prioritized action list.

### Summary Generation

In production, an LLM generates the briefing from the profile + score context. Here we use **template-based generation** to show the pattern.


In [ ]:
def generate_summary(record: CRMRecord, profile: CompanyProfile,
                     score: LeadScore) -> CompanySummary:
    """Layer 3 Tool: generate an executive briefing for a lead."""

    # Headline
    headline = (
        f"{profile.name} — {profile.sub_industry} | "
        f"{profile.employee_count} employees | "
        f"{profile.revenue_estimate} revenue | "
        f"Score: {score.total_score}/100 ({score.tier})"
    )

    # Briefing paragraph
    briefing_parts = [
        f"{profile.name} is a {profile.funding_stage.replace('_', ' ')} "
        f"{profile.sub_industry.lower()} company based in {profile.headquarters}.",
        profile.description,
    ]
    if profile.recent_news:
        briefing_parts.append(f"Recent news: {profile.recent_news[0]}.")
    briefing_parts.append(
        f"The lead ({record.contact_name}, {record.contact_title}) "
        f"is currently in the '{record.stage.value}' stage "
        f"with an estimated deal value of ${record.deal_value:,.0f}."
    )
    briefing = " ".join(briefing_parts)

    # Key facts
    key_facts = [
        f"Industry: {profile.industry} / {profile.sub_industry}",
        f"Size: {profile.employee_count} employees, {profile.revenue_estimate} revenue",
        f"Founded: {profile.founded_year} | HQ: {profile.headquarters}",
        f"Funding: {profile.funding_stage.replace('_', ' ')}",
        f"Tech stack: {', '.join(profile.tech_stack[:4])}",
        f"Competitors: {', '.join(profile.competitors[:3])}",
    ]

    # Talking points (derived from notes and profile)
    talking_points = []
    if "compliance" in record.notes.lower() or "soc2" in record.notes.lower():
        talking_points.append("Address compliance concerns early — mention our SOC 2 and GDPR certifications")
    if "competitor" in record.notes.lower() or "competing" in record.notes.lower():
        talking_points.append("Prepare competitive differentiation — they are evaluating alternatives")
    if "budget" in record.notes.lower():
        talking_points.append("Budget is being discussed — align proposal with their fiscal calendar")
    if "demo" in record.notes.lower():
        talking_points.append("Reschedule demo — this is their next expected interaction")
    if "partnership" in record.notes.lower():
        talking_points.append("Explore partnership angle — value extends beyond a single sale")
    if profile.employee_count > 500:
        talking_points.append(f"Enterprise account ({profile.employee_count} employees) — expect multi-stakeholder process")
    if not talking_points:
        talking_points.append("Standard intro — highlight core value proposition for their industry")
    talking_points.append(f"Reference recent news: {profile.recent_news[0][:50]}..." if profile.recent_news else "No recent news to reference")

    # Risk factors
    risk_factors = []
    days_since = (date(2024, 6, 15) - date.fromisoformat(record.last_contact_date)).days
    if days_since > 30:
        risk_factors.append(f"Contact is stale ({days_since} days since last interaction)")
    if record.stage == LeadStage.CLOSED_LOST:
        risk_factors.append("Previously closed-lost — re-engagement may face resistance")
    if "no response" in record.notes.lower():
        risk_factors.append("Contact has gone silent — may need a different champion")
    if "legal" in record.notes.lower() or "procurement" in record.notes.lower():
        risk_factors.append("Legal/procurement involved — expect slower timeline")
    if "discount" in record.notes.lower():
        risk_factors.append("Price sensitivity — be prepared to justify value")
    if profile.employee_count > 2000:
        risk_factors.append("Large enterprise — long sales cycle expected")
    if not risk_factors:
        risk_factors.append("No significant risk factors identified")

    return CompanySummary(
        headline=headline,
        briefing=briefing,
        key_facts=key_facts,
        talking_points=talking_points,
        risk_factors=risk_factors,
    )


# Test
test_summary = generate_summary(CRM_RECORDS[0], test_profile, test_score)
print("EXECUTIVE BRIEFING:")
print(f"  {test_summary.headline}")
print()
print(textwrap.fill(test_summary.briefing, width=80))
print()
print("Key facts:")
for f in test_summary.key_facts:
    print(f"  - {f}")
print()
print("Talking points:")
for t in test_summary.talking_points:
    print(f"  * {t}")
print()
print("Risk factors:")
for r in test_summary.risk_factors:
    print(f"  ! {r}")

In [ ]:
# ── Action templates ──
ACTION_TEMPLATES = {
    ActionType.SEND_EMAIL: "Hi {name},\n\nI wanted to follow up on {topic}. {body}\n\nBest,\n[Your Name]",
    ActionType.SCHEDULE_CALL: "Reach out to {name} to schedule a {duration}-minute call about {topic}.",
    ActionType.SEND_CASE_STUDY: "Send {name} our case study on {industry} — specifically the {use_case} results.",
    ActionType.SCHEDULE_DEMO: "Schedule a personalized demo for {name} focusing on {features}.",
    ActionType.FOLLOW_UP: "Follow up with {name} regarding {topic}. Reference {context}.",
    ActionType.ESCALATE: "Escalate {company} deal to sales leadership. Reason: {reason}.",
    ActionType.NURTURE: "Add {name} to the {industry} nurture drip campaign.",
    ActionType.RE_ENGAGE: "Re-engage {name} at {company}. Their contract with competitor is approaching renewal.",
    ActionType.SEND_PROPOSAL: "Prepare and send a formal proposal to {name} for ${value:,.0f}.",
    ActionType.INTRO_PARTNER: "Introduce {name} to our {partner_type} partner for {reason}.",
}


def recommend_actions(record: CRMRecord, profile: CompanyProfile,
                      score: LeadScore) -> list[SuggestedAction]:
    """Layer 3 Tool: recommend prioritized next actions."""
    actions: list[SuggestedAction] = []
    notes_lower = record.notes.lower()

    days_since = (date(2024, 6, 15) - date.fromisoformat(record.last_contact_date)).days

    # ── Stage-based primary actions ──
    if record.stage == LeadStage.NEW:
        template = ACTION_TEMPLATES[ActionType.SEND_EMAIL].format(
            name=record.contact_name.split()[0],
            topic="your recent interest in our platform",
            body=f"I noticed you {('downloaded our whitepaper' if record.source == 'inbound' else 'visited our booth at the conference')}. "
                 f"Given {profile.name}'s work in {profile.sub_industry.lower()}, I think our solution could help with [specific value prop].",
        )
        actions.append(SuggestedAction(
            action_type=ActionType.SEND_EMAIL,
            priority=1 if score.tier == "hot" else 2,
            description=f"Send introductory email to {record.contact_name}",
            reasoning=f"New lead with {record.source} source. Score: {score.total_score}/100.",
            deadline_days=1 if score.tier == "hot" else 3,
            template=template,
        ))

    elif record.stage == LeadStage.CONTACTED:
        if days_since > 14:
            actions.append(SuggestedAction(
                action_type=ActionType.FOLLOW_UP,
                priority=1,
                description=f"Follow up with {record.contact_name} ({days_since}d since last contact)",
                reasoning=f"Contact has gone silent for {days_since} days. Risk of losing momentum.",
                deadline_days=1,
                template=ACTION_TEMPLATES[ActionType.FOLLOW_UP].format(
                    name=record.contact_name.split()[0],
                    topic="our conversation about your data infrastructure needs",
                    context="their initial interest in our platform",
                ),
            ))
        if "compliance" in notes_lower or "soc2" in notes_lower:
            actions.append(SuggestedAction(
                action_type=ActionType.SEND_CASE_STUDY,
                priority=2,
                description=f"Send compliance case study to {record.contact_name}",
                reasoning="Compliance concerns detected — address proactively with proof points.",
                deadline_days=2,
                template=ACTION_TEMPLATES[ActionType.SEND_CASE_STUDY].format(
                    name=record.contact_name.split()[0],
                    industry=profile.industry,
                    use_case="regulatory compliance and audit readiness",
                ),
            ))

    elif record.stage == LeadStage.QUALIFIED:
        actions.append(SuggestedAction(
            action_type=ActionType.SCHEDULE_DEMO,
            priority=1,
            description=f"Schedule demo for {record.contact_name}",
            reasoning=f"Qualified lead with ${record.deal_value:,.0f} deal value. Demo is the natural next step.",
            deadline_days=3,
            template=ACTION_TEMPLATES[ActionType.SCHEDULE_DEMO].format(
                name=record.contact_name.split()[0],
                features=f"{profile.sub_industry.lower()} use cases and ROI calculator",
            ),
        ))

    elif record.stage == LeadStage.PROPOSAL:
        actions.append(SuggestedAction(
            action_type=ActionType.FOLLOW_UP,
            priority=1,
            description=f"Follow up on proposal sent to {record.company_name}",
            reasoning="Proposal is pending. Keep momentum and address any questions.",
            deadline_days=2,
            template=ACTION_TEMPLATES[ActionType.FOLLOW_UP].format(
                name=record.contact_name.split()[0],
                topic="the proposal we sent on June 5th",
                context="procurement timeline and any outstanding questions",
            ),
        ))

    elif record.stage == LeadStage.NEGOTIATION:
        if "discount" in notes_lower:
            actions.append(SuggestedAction(
                action_type=ActionType.ESCALATE,
                priority=1,
                description=f"Escalate pricing discussion for {record.company_name}",
                reasoning="Discount request detected. Need approval for modified terms.",
                deadline_days=1,
                template=ACTION_TEMPLATES[ActionType.ESCALATE].format(
                    company=record.company_name,
                    reason=f"requesting 20% discount on ${record.deal_value:,.0f} annual deal",
                ),
            ))
        actions.append(SuggestedAction(
            action_type=ActionType.SEND_PROPOSAL,
            priority=2,
            description=f"Send revised proposal to {record.company_name}",
            reasoning="Negotiation stage — formalize terms with an updated proposal.",
            deadline_days=3,
            template=ACTION_TEMPLATES[ActionType.SEND_PROPOSAL].format(
                name=record.contact_name.split()[0],
                value=record.deal_value,
            ),
        ))

    elif record.stage == LeadStage.CLOSED_LOST:
        if "reconsider" in notes_lower or "renewal" in notes_lower:
            actions.append(SuggestedAction(
                action_type=ActionType.RE_ENGAGE,
                priority=3,
                description=f"Re-engage {record.contact_name} at {record.company_name}",
                reasoning="Closed-lost but mentioned possible reconsideration. Nurture until renewal window.",
                deadline_days=30,
                template=ACTION_TEMPLATES[ActionType.RE_ENGAGE].format(
                    name=record.contact_name.split()[0],
                    company=record.company_name,
                ),
            ))
        actions.append(SuggestedAction(
            action_type=ActionType.NURTURE,
            priority=4,
            description=f"Add {record.contact_name} to nurture campaign",
            reasoning="Closed-lost. Keep relationship warm with relevant content.",
            deadline_days=7,
            template=ACTION_TEMPLATES[ActionType.NURTURE].format(
                name=record.contact_name.split()[0],
                industry=profile.industry,
            ),
        ))

    # ── Cross-stage actions based on signals ──
    if "partnership" in notes_lower and record.stage != LeadStage.CLOSED_LOST:
        actions.append(SuggestedAction(
            action_type=ActionType.INTRO_PARTNER,
            priority=2,
            description=f"Explore strategic partnership with {record.company_name}",
            reasoning="Partnership potential mentioned. Could multiply deal value.",
            deadline_days=5,
            template=ACTION_TEMPLATES[ActionType.INTRO_PARTNER].format(
                name=record.contact_name.split()[0],
                partner_type="channel",
                reason="potential strategic partnership and portfolio-wide rollout",
            ),
        ))

    if score.tier == "hot" and record.stage not in (LeadStage.CLOSED_WON, LeadStage.CLOSED_LOST):
        actions.append(SuggestedAction(
            action_type=ActionType.SCHEDULE_CALL,
            priority=1,
            description=f"Schedule call with {record.contact_name} (hot lead)",
            reasoning=f"Score {score.total_score}/100 — prioritize personal outreach.",
            deadline_days=1,
            template=ACTION_TEMPLATES[ActionType.SCHEDULE_CALL].format(
                name=record.contact_name.split()[0],
                duration="30",
                topic=f"how we can help {profile.name} with {profile.sub_industry.lower()}",
            ),
        ))

    actions.sort(key=lambda a: a.priority)
    return actions


# Test
test_actions = recommend_actions(CRM_RECORDS[0], test_profile, test_score)
print(f"Recommended actions for {CRM_RECORDS[0].contact_name}:\n")
for a in test_actions:
    print(f"  P{a.priority}  [{a.action_type}]  {a.description}")
    print(f"       Deadline: {a.deadline_days} day(s)")
    print(f"       Reason: {a.reasoning}")
    print()

## 7. Assembling the Full Agent

In [ ]:
class CRMEnrichmentAgent:
    """End-to-end CRM enrichment agent.

    Data flow:
      CRM Record → Layer 1 (Lookup) → Layer 2 (Score) → Layer 3 (Generate) → EnrichedRecord
    """

    def __init__(self, tools: ToolRegistry):
        self.tools = tools
        self._results: list[EnrichedRecord] = []

    def enrich(self, record: CRMRecord, verbose: bool = False) -> EnrichedRecord:
        """Enrich a single CRM record through all layers."""

        if verbose:
            print(f"[Layer 0] INPUT:  {record.lead_id} — {record.contact_name} @ {record.company_name}")

        # Layer 1: Lookup
        profile = self.tools.lookup_company(record.company_domain)
        if verbose:
            print(f"[Layer 1] LOOKUP: {profile.name} | {profile.industry} | "
                  f"{profile.employee_count} emp | {profile.funding_stage}")
            print(f"          News:   {len(profile.recent_news)} items | "
                  f"Stack: {', '.join(profile.tech_stack[:3])}")

        # Layer 2: Score
        score = score_lead(record, profile)
        if verbose:
            print(f"[Layer 2] SCORE:  {score.total_score}/100 ({score.tier}) "
                  f"[fit={score.fit_score} eng={score.engagement_score} "
                  f"time={score.timing_score}]")

        # Layer 3: Generate
        summary = generate_summary(record, profile, score)
        actions = recommend_actions(record, profile, score)
        if verbose:
            print(f"[Layer 3] GEN:    {len(summary.talking_points)} talking points | "
                  f"{len(actions)} actions")
            print(f"          Top action: [{actions[0].action_type}] {actions[0].description[:50]}" if actions else "          No actions")

        result = EnrichedRecord(
            lead=record,
            profile=profile,
            score=score,
            summary=summary,
            actions=actions,
        )
        self._results.append(result)
        return result

    def enrich_batch(self, records: list[CRMRecord],
                     verbose: bool = False) -> list[EnrichedRecord]:
        """Enrich multiple CRM records."""
        results = []
        for r in records:
            results.append(self.enrich(r, verbose=verbose))
            if verbose:
                print()
        return results

    @property
    def results(self):
        return self._results

    def summary_dataframe(self) -> pd.DataFrame:
        rows = []
        for r in self._results:
            rows.append({
                "lead_id": r.lead.lead_id,
                "contact": r.lead.contact_name,
                "company": r.lead.company_name,
                "industry": r.profile.industry,
                "employees": r.profile.employee_count,
                "stage": str(r.lead.stage),
                "deal_value": r.lead.deal_value,
                "score": r.score.total_score,
                "tier": r.score.tier,
                "fit": r.score.fit_score,
                "engagement": r.score.engagement_score,
                "timing": r.score.timing_score,
                "top_action": str(r.actions[0].action_type) if r.actions else "none",
                "n_actions": len(r.actions),
                "n_risks": len(r.summary.risk_factors),
            })
        return pd.DataFrame(rows)


agent = CRMEnrichmentAgent(tools)
print("CRMEnrichmentAgent ready.")

## 8. Running the Agent — Full Enrichment

In [ ]:
all_enriched = agent.enrich_batch(CRM_RECORDS)

print(f"Enriched {len(all_enriched)} records.\n")
df = agent.summary_dataframe()
print(df[["lead_id", "company", "industry", "stage", "score", "tier", "top_action"]].to_string(index=False))

## 9. Verbose Walkthrough — Single Record

In [ ]:
wt_agent = CRMEnrichmentAgent(ToolRegistry())

print("VERBOSE ENRICHMENT WALKTHROUGH")
print("=" * 70)
result = wt_agent.enrich(CRM_RECORDS[3], verbose=True)  # FinServe Capital

print(f"\n{'='*70}")
print("EXECUTIVE BRIEFING:")
print(result.summary.headline)
print()
print(textwrap.fill(result.summary.briefing, width=75))
print()
print("Talking points:")
for t in result.summary.talking_points:
    print(f"  * {t}")
print()
print("Risk factors:")
for r in result.summary.risk_factors:
    print(f"  ! {r}")
print()
print("Recommended actions:")
for a in result.actions:
    print(f"  P{a.priority} [{a.action_type}] {a.description}")
    print(f"     Due: {a.deadline_days} day(s) | {a.reasoning}")

## 10. Data Flow & Tool Layers — Deep Dive

### The Four Layers Explained

**Layer 0 — Input** is the raw CRM record. It contains only what the sales rep entered: name, company, stage, notes. It is *sparse* — missing industry, employee count, tech stack, and context.

**Layer 1 — Lookup** enriches the record with *external knowledge*. Each tool fetches one type of data:
- `lookup_company(domain)` → company profile (industry, size, funding)
- `lookup_news(domain)` → recent events and triggers
- `lookup_tech_stack(domain)` → technologies used

Layer 1 tools are **pure data retrieval** — they add information but make no judgments.

**Layer 2 — Analyze** takes the enriched data and produces *computed signals*:
- `score_lead(record, profile)` → fit + engagement + timing scores
- Each score component is independently testable

Layer 2 tools are **pure computation** — deterministic functions with no side effects.

**Layer 3 — Generate** produces *human-readable outputs*:
- `generate_summary(record, profile, score)` → executive briefing
- `recommend_actions(record, profile, score)` → prioritized next steps

Layer 3 tools are **template-based** (or LLM-based in production) generators.

### Why This Layering Matters

```python
# Each layer can be tested independently:
profile = tools.lookup_company("techflow.io")  # Layer 1
assert profile.industry == "Technology"

score = score_lead(record, profile)             # Layer 2
assert 0 <= score.total_score <= 100

summary = generate_summary(record, profile, score)  # Layer 3
assert len(summary.talking_points) > 0

# And the full pipeline composes cleanly:
enriched = agent.enrich(record)                 # All layers
assert enriched.score.tier in ("hot", "warm", "cold")
```

### Swapping Layers in Production

```python
# Swap Layer 1: Mock → Real API
class ClearbitLookup(ToolRegistry):
    def lookup_company(self, domain: str) -> CompanyProfile:
        response = clearbit.Company.find(domain=domain)
        return CompanyProfile(
            name=response.name,
            industry=response.category.industry,
            employee_count=response.metrics.employees,
            ...
        )

# Swap Layer 3: Templates → LLM
class LLMSummaryGenerator:
    def generate_summary(self, record, profile, score) -> CompanySummary:
        prompt = f"Write a 3-sentence executive briefing for {profile.name}..."
        response = openai.chat.completions.create(
            model="gpt-4o", messages=[{"role": "user", "content": prompt}]
        )
        return CompanySummary(briefing=response.choices[0].message.content, ...)
```

The agent code doesn't change — only the tool implementations do.


## 11. Enrichment Analytics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Chart 1: Score breakdown stacked bar
ax = axes[0, 0]
df_sorted = df.sort_values("score", ascending=True)
ax.barh(df_sorted["company"], df_sorted["fit"], color="#3498db", alpha=0.8, label="Fit")
ax.barh(df_sorted["company"], df_sorted["engagement"],
        left=df_sorted["fit"], color="#2ecc71", alpha=0.8, label="Engagement")
ax.barh(df_sorted["company"], df_sorted["timing"],
        left=df_sorted["fit"] + df_sorted["engagement"],
        color="#f39c12", alpha=0.8, label="Timing")
ax.set_xlabel("Score")
ax.set_title("Lead Score Breakdown")
ax.legend(fontsize=8, loc="lower right")
for i, (_, row) in enumerate(df_sorted.iterrows()):
    ax.text(row["score"] + 1, i, f"{row['score']}", va="center", fontsize=8)

# Chart 2: Tier distribution
ax2 = axes[0, 1]
tier_counts = df["tier"].value_counts()
tier_colors = {"hot": "#e74c3c", "warm": "#f39c12", "cold": "#3498db"}
ax2.pie(tier_counts.values, labels=tier_counts.index,
        colors=[tier_colors.get(t, "#95a5a6") for t in tier_counts.index],
        autopct="%1.0f%%", startangle=90, textprops={"fontsize": 10})
ax2.set_title("Lead Tier Distribution")

# Chart 3: Deal value vs score scatter
ax3 = axes[1, 0]
tier_color_map = df["tier"].map(tier_colors)
ax3.scatter(df["deal_value"] / 1000, df["score"], c=tier_color_map,
            s=df["employees"].clip(upper=500) / 5 + 30,
            alpha=0.7, edgecolors="black", linewidth=0.5)
for _, row in df.iterrows():
    ax3.annotate(row["company"][:12], (row["deal_value"] / 1000, row["score"]),
                 fontsize=7, ha="left", va="bottom")
ax3.set_xlabel("Deal Value ($K)")
ax3.set_ylabel("Lead Score")
ax3.set_title("Deal Value vs. Lead Score (size = company size)")

# Chart 4: Top recommended actions
ax4 = axes[1, 1]
all_action_types = [str(a.action_type) for r in agent.results for a in r.actions]
action_counts = Counter(all_action_types).most_common(8)
if action_counts:
    ax4.barh([a[0].replace("_", " ") for a in action_counts],
             [a[1] for a in action_counts],
             color="#9467bd", alpha=0.8)
    ax4.set_xlabel("Frequency")
    ax4.set_title("Most Recommended Actions")

plt.suptitle("CRM Enrichment Agent — Analytics", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 12. Prioritized Action Board

In [ ]:
print("ACTION BOARD — Sorted by Priority")
print("=" * 80)

all_actions = []
for result in agent.results:
    for action in result.actions:
        all_actions.append({
            "priority": action.priority,
            "lead": result.lead.lead_id,
            "company": result.lead.company_name[:20],
            "tier": result.score.tier,
            "action": str(action.action_type),
            "description": action.description[:50],
            "deadline": f"{action.deadline_days}d",
            "deal_value": result.lead.deal_value,
        })

action_df = pd.DataFrame(all_actions).sort_values(["priority", "deal_value"],
                                                     ascending=[True, False])
print(action_df[["priority", "lead", "company", "tier", "action", "deadline"]].to_string(index=False))

print(f"\nTotal actions: {len(action_df)}")
print(f"P1 (urgent):  {(action_df['priority'] == 1).sum()}")
print(f"P2:           {(action_df['priority'] == 2).sum()}")
print(f"P3+:          {(action_df['priority'] >= 3).sum()}")

## 13. Draft Response Preview

In [ ]:
# Show draft outreach templates for hot leads
hot_leads = [r for r in agent.results if r.score.tier == "hot"]

for result in hot_leads[:3]:
    primary_action = result.actions[0] if result.actions else None
    print(f"{'='*60}")
    print(f"{result.lead.lead_id}: {result.lead.contact_name} @ {result.lead.company_name}")
    print(f"Score: {result.score.total_score}/100 ({result.score.tier}) | "
          f"Deal: ${result.lead.deal_value:,.0f}")
    print(f"{'='*60}")
    if primary_action and primary_action.template:
        print(f"\nAction: [{primary_action.action_type}] {primary_action.description}")
        print(f"Deadline: {primary_action.deadline_days} day(s)\n")
        print("--- DRAFT ---")
        print(primary_action.template)
        print("--- END ---")
    print()

## 14. Pipeline Metrics & Tool Call Analysis

In [ ]:
# Tool call analysis from the registry
call_df = pd.DataFrame(tools.call_log)
if not call_df.empty:
    print("TOOL CALL LOG:")
    print(f"  Total calls: {len(call_df)}")
    print(f"  By tool:")
    for tool, count in call_df["tool"].value_counts().items():
        print(f"    {tool}: {count}")
    print(f"  Success rate: {call_df['success'].mean():.0%}")
    print()

# Enrichment summary
print("ENRICHMENT SUMMARY:")
print(f"  Records enriched: {len(agent.results)}")
print(f"  Avg score:        {df['score'].mean():.1f}/100")
print(f"  Hot leads:        {(df['tier'] == 'hot').sum()}")
print(f"  Warm leads:       {(df['tier'] == 'warm').sum()}")
print(f"  Cold leads:       {(df['tier'] == 'cold').sum()}")
print(f"  Total deal value: ${df['deal_value'].sum():,.0f}")
print(f"  Hot pipeline:     ${df[df['tier']=='hot']['deal_value'].sum():,.0f}")
print(f"  Total actions:    {df['n_actions'].sum()}")
print(f"  Avg actions/lead: {df['n_actions'].mean():.1f}")

In [ ]:
# Pipeline visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Chart 1: Tool call funnel
ax = axes[0]
layer_labels = ["L0: Input", "L1: Lookup", "L2: Score", "L3: Generate", "L4: Output"]
layer_counts = [len(CRM_RECORDS), len(call_df) if not call_df.empty else 0,
                len(agent.results), len(agent.results), len(agent.results)]
ax.barh(layer_labels[::-1], layer_counts[::-1],
        color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"][::-1], alpha=0.8)
ax.set_xlabel("Operations")
ax.set_title("Pipeline Layer Activity")

# Chart 2: Score distribution histogram
ax2 = axes[1]
ax2.hist(df["score"], bins=10, color="#3498db", alpha=0.7, edgecolor="black")
ax2.axvline(df["score"].mean(), color="red", linestyle="--", label=f"Mean: {df['score'].mean():.0f}")
ax2.axvline(65, color="orange", linestyle=":", label="Hot threshold: 65")
ax2.axvline(40, color="blue", linestyle=":", label="Warm threshold: 40")
ax2.set_xlabel("Lead Score")
ax2.set_ylabel("Count")
ax2.set_title("Score Distribution")
ax2.legend(fontsize=8)

# Chart 3: Industry diversity
ax3 = axes[2]
ind_counts = df["industry"].value_counts()
ax3.pie(ind_counts.values, labels=ind_counts.index, autopct="%1.0f%%",
        startangle=90, textprops={"fontsize": 9})
ax3.set_title("Leads by Industry")

plt.suptitle("CRM Enrichment Pipeline Metrics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 15. Agent Evaluation

In [ ]:
EXPECTED = {
    "LEAD-001": {"tier": "hot",  "has_demo_action": True},
    "LEAD-002": {"tier": "warm", "has_email_action": True},
    "LEAD-003": {"tier": "hot",  "has_followup": True},
    "LEAD-004": {"tier": "warm", "has_followup": True},
    "LEAD-005": {"tier": "warm", "has_email_action": True},
    "LEAD-006": {"tier": "hot",  "has_discount_escalation": False},
    "LEAD-007": {"tier": "warm", "has_demo_action": True},
    "LEAD-008": {"tier": "cold", "has_followup": True},
    "LEAD-009": {"tier": "cold", "has_reengage": True},
    "LEAD-010": {"tier": "hot",  "has_partnership_action": True},
}

eval_rows = []
for r in agent.results:
    exp = EXPECTED.get(r.lead.lead_id, {})
    action_types = {str(a.action_type) for a in r.actions}

    tier_ok = r.score.tier == exp.get("tier", r.score.tier)
    action_ok = True
    if exp.get("has_demo_action"):
        action_ok = action_ok and "schedule_demo" in action_types
    if exp.get("has_email_action"):
        action_ok = action_ok and "send_email" in action_types
    if exp.get("has_followup"):
        action_ok = action_ok and "follow_up" in action_types
    if exp.get("has_reengage"):
        action_ok = action_ok and "re_engage" in action_types
    if exp.get("has_partnership_action"):
        action_ok = action_ok and "introduce_partner" in action_types
    if exp.get("has_discount_escalation"):
        action_ok = action_ok and "escalate" in action_types

    has_summary = bool(r.summary.briefing)
    has_score = 0 <= r.score.total_score <= 100
    has_actions = len(r.actions) > 0

    all_pass = tier_ok and action_ok and has_summary and has_score and has_actions

    eval_rows.append({
        "lead": r.lead.lead_id,
        "tier_ok": tier_ok,
        "action_ok": action_ok,
        "summary_ok": has_summary,
        "score_ok": has_score,
        "actions_present": has_actions,
        "passed": all_pass,
    })

eval_df = pd.DataFrame(eval_rows)
print("EVALUATION RESULTS")
print("=" * 75)
for _, row in eval_df.iterrows():
    checks = " ".join(
        f"{k}={'ok' if v else 'FAIL'}"
        for k, v in row.items()
        if k not in ("lead", "passed")
    )
    icon = "pass" if row["passed"] else "FAIL"
    print(f"  [{icon:4s}]  {row['lead']}  {checks}")

print(f"\nOverall: {eval_df['passed'].sum()}/{len(eval_df)} "
      f"({eval_df['passed'].mean():.0%})")

## 16. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "crm_enrichment_agent",
    "records_enriched": len(agent.results),
    "tool_calls": len(tools.call_log),
    "avg_score": float(df["score"].mean()),
    "tier_distribution": dict(df["tier"].value_counts()),
    "total_pipeline_value": float(df["deal_value"].sum()),
    "hot_pipeline_value": float(df[df["tier"] == "hot"]["deal_value"].sum()),
    "total_actions_generated": int(df["n_actions"].sum()),
    "layers": {
        "layer_0_input": "Raw CRM records (CSV/JSON)",
        "layer_1_lookup": "Company profile, news, tech stack",
        "layer_2_analyze": "Lead scoring (fit + engagement + timing)",
        "layer_3_generate": "Summary briefing + action recommendations",
        "layer_4_output": "EnrichedRecord (typed dataclass → JSON)",
    },
    "evaluation_pass_rate": float(eval_df["passed"].mean()),
}

log_path = ARTIFACT_DIR / "crm_enrichment_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")

# Save enriched records
records_path = ARTIFACT_DIR / "enriched_records.json"
records_data = [r.to_dict() for r in agent.results]
records_path.write_text(json.dumps(records_data, indent=2, default=str), encoding="utf-8")

print(f"Saved: {log_path}")
print(f"Saved: {records_path}")
print(f"\nFinal stats:")
print(f"  Enriched:     {log['records_enriched']} records")
print(f"  Tool calls:   {log['tool_calls']}")
print(f"  Avg score:    {log['avg_score']:.1f}/100")
print(f"  Hot pipeline: ${log['hot_pipeline_value']:,.0f}")
print(f"  Evaluation:   {log['evaluation_pass_rate']:.0%}")

## 17. Key Takeaways

### What We Built
- A **CRM enrichment agent** that transforms sparse lead records into rich, actionable briefings
- **4-layer tool architecture**: Input → Lookup → Analyze → Generate
- **3 Layer-1 lookup tools** (company profile, news, tech stack) with graceful degradation
- **Multi-factor lead scoring** (fit + engagement + timing → 0–100 score)
- **Executive briefings** with talking points, risk factors, and key facts
- **Prioritized action recommendations** with draft templates and deadlines
- **10 realistic CRM records** across 6 industries and 7 pipeline stages

### Data Flow Principles
1. **Each layer transforms data** — raw → enriched → scored → summarized
2. **Typed interfaces between layers** — `CRMRecord → CompanyProfile → LeadScore → CompanySummary`
3. **Independent testability** — test each layer in isolation
4. **Composable tools** — reuse lookup tools in other agents
5. **Graceful degradation** — missing data returns defaults, never crashes

### Tool Layer Design
| Layer | Purpose | Input | Output | Side effects |
|---|---|---|---|---|
| **0: Input** | Ingest | CSV/JSON | `CRMRecord` | None |
| **1: Lookup** | Enrich | Domain | `CompanyProfile` | API calls |
| **2: Analyze** | Score | Record + Profile | `LeadScore` | None (pure) |
| **3: Generate** | Write | All above | Summary + Actions | None (pure) |
| **4: Output** | Deliver | `EnrichedRecord` | JSON / Dashboard | CRM write |

### Production Enhancements
| Enhancement | Purpose |
|---|---|
| **Clearbit / Apollo** | Real company data lookup |
| **LLM summaries** | GPT-4o for personalized briefings |
| **Intent detection** | Parse CRM notes for buying signals via NLP |
| **CRM write-back** | Push scores and actions back to Salesforce/HubSpot |
| **Batch scheduling** | Enrich new leads on a nightly cron job |
| **A/B testing** | Test different action templates for reply rates |
| **Feedback loop** | Sales reps rate action quality → improve scoring |
